# FACULTY SOLUTION — Week 6→8 cohort window 2: Cohort feedback window 2 — Gulf Coast capacity additions
Computes every answer from the canonical CSVs and asserts it against the golden fixtures.

In [ ]:
import pandas as pd, numpy as np, math, json, itertools
d = {s: pd.read_csv(f'../data/{s}.csv') for s in ['window2_params', 'cohort_states', 'baton_rouge_margin', 'week8_link', 'worked_example_window']}

In [ ]:

def compute(d):
    w = {x.parameter: x.value for x in d['window2_params'].itertuples()}
    b = {x.parameter: x.value for x in d['baton_rouge_margin'].itertuples()}
    k = {x.parameter: x.value for x in d['week8_link'].itertuples()}
    def shift(s): return -w['down_slope'] * (s - w['pivot_share']) if s > w['pivot_share'] else w['up_slope'] * (w['pivot_share'] - s)
    r = {'br_margin_baseline': b['gc_crack_base'] + b['complexity'] - b['opex']}
    pcts = []
    for c in d['cohort_states'].itertuples():
        r['crack_shift_' + c.state] = shift(c.gc_capacity_add_share)
        r['br_margin_' + c.state] = r['br_margin_baseline'] + r['crack_shift_' + c.state]
        r['br_margin_change_pct_' + c.state] = r['crack_shift_' + c.state] / r['br_margin_baseline']
        pcts.append(abs(r['br_margin_change_pct_' + c.state]))
    r['max_abs_change_pct'] = max(pcts)
    r['bound_pct'] = w['bound_pct']
    r['worst_case_crack_with_opec_holds_full'] = b['gc_crack_base'] + k['crack_coefficient'] * k['holds_full_delta_wti'] + r['crack_shift_all_add']
    r['worst_case_br_margin_with_opec_holds_full'] = r['worst_case_crack_with_opec_holds_full'] + b['complexity'] - b['opex']
    return r


In [ ]:
fx = json.load(open('../fixtures/window2_golden.json'))
r = compute(d)
bad = []
for k, v in fx['results'].items():
    a = r[k]
    ok = (str(a) == str(v)) if isinstance(v, str) else math.isclose(float(a), float(v), rel_tol=1e-3, abs_tol=1e-5)
    print(('OK  ' if ok else 'DIFF'), f'{k:36s}', a)
    if not ok: bad.append(k)
assert not bad, bad
print('FACULTY NOTEBOOK REPRODUCES GOLDEN FIXTURES: PASS')